# [Module 02 -- Fundamentals] Agents Deep Dive

> **MLCourse -- Agentic AI -- CrewAI Fundamentals**

> An `Agent` is the core worker unit in CrewAI. This module explores every
> parameter you can attach to an agent: role, goal, backstory, LLM assignment,
> iteration limits, delegation, verbosity, and reasoning. We also compare three
> ways to define agent configurations side by side.

## What you'll learn

- Every `Agent()` parameter and what it controls.
- How to assign different LLMs (including different Ollama models) to agents.
- `max_iter` -- why it matters and how to tune it.
- `allow_delegation` -- agent-to-agent handoff.
- `verbose` vs `reasoning` -- debugging output modes.
- Three config approaches: Python constructor, JSON dict, YAML file.

In [ ]:
# --- Standard library imports -------------------------------------------------
import json                          # For JSON-based agent config.
import os                            # Environment variable access.
from pathlib import Path             # Track-root walk pattern.

# --- Third-party imports ------------------------------------------------------
from dotenv import load_dotenv

# Walk up to the track root and load .env.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("Setup complete. Track root:", TRACK)

## 1. Imports and LLM setup

We need `Agent` from crewai and a LangChain chat model. CrewAI accepts any
LangChain-compatible LLM -- we use `ChatOllama` for local, free inference.

In [ ]:
try:
    from crewai import Agent
    from langchain_ollama import ChatOllama
    print("[OK] crewai and ChatOllama imported.")
except ImportError as e:
    print("[ERROR] Missing dependency. Run:  pip install crewai crewai[tools] langchain-ollama")
    print("        Detail:", e)

In [ ]:
LLM_MODEL = "llama3.1:8b"
llm = ChatOllama(model=LLM_MODEL)

# Quick connectivity check.
try:
    resp = llm.invoke("Reply with just the word OK.")
    print("[OK] Ollama responding:", resp.content[:40])
except Exception as e:
    print("[WARN] Ollama unreachable. Start Ollama, then: ollama pull", LLM_MODEL)
    print("       Detail:", e)

## 2. The six core Agent parameters

| Parameter          | Type          | Default | Purpose                                      |
|--------------------|---------------|---------|----------------------------------------------|
| `role`             | str           | --      | Short title for the agent.                   |
| `goal`             | str           | --      | One-sentence objective.                      |
| `backstory`        | str           | --      | Context paragraph that shapes LLM persona.   |
| `llm`              | LLM or str    | default | The language model powering this agent.      |
| `tools`            | list          | []      | Tool objects the agent can call.             |
| `allow_delegation` | bool          | True    | Can this agent hand tasks to other agents?   |
| `max_iter`         | int           | 15      | Max reasoning iterations before forced stop. |
| `verbose`          | bool          | True    | Print detailed execution logs.               |
| `reasoning`        | bool          | False   | Enable explicit chain-of-thought logging.    |

Let us create agents that exercise each parameter.

In [ ]:
# Agent with full explicit parameters.
researcher = Agent(
    role="Senior Research Analyst",
    goal="Uncover cutting-edge developments in AI.",
    backstory=(
        "You are a veteran researcher at a top tech think tank. "
        "Your expertise lies in identifying emerging trends."
    ),
    llm=llm,                          # ChatOllama instance.
    tools=[],                         # No tools yet -- covered in Module 04.
    allow_delegation=False,           # Works independently.
    max_iter=5,                       # Stop after 5 reasoning loops if no answer.
    verbose=True,                     # Print every step.
    reasoning=False,                  # No explicit chain-of-thought logging.
)

print("Agent created:", researcher.role)
print("  max_iter       :", researcher.max_iter)
print("  allow_delegation:", researcher.allow_delegation)
print("  verbose        :", researcher.verbose)

## 3. LLM assignment -- different models for different agents

CrewAI lets you assign a **different LLM to each agent**. This is powerful:
use a fast, small model for simple tasks and a larger model for complex
reasoning. You can also pass a model tag string and CrewAI resolves it.

Below we create two agents with different Ollama models (both guarded).

In [ ]:
# Agent A -- fast model for quick responses.
llm_fast = ChatOllama(model="llama3.1:8b")

agent_fast = Agent(
    role="Quick Summarizer",
    goal="Produce a one-sentence summary.",
    backstory="You are fast and concise.",
    llm=llm_fast,
    allow_delegation=False,
    verbose=False,
)

# Agent B -- could be a larger model (e.g. llama3.1:8b) for deep reasoning.
# For demonstration we use the same model; swap the tag for a larger one if available.
llm_deep = ChatOllama(model="llama3.1:8b")

agent_deep = Agent(
    role="Deep Analyst",
    goal="Provide a thorough multi-paragraph analysis.",
    backstory="You are thorough and methodical.",
    llm=llm_deep,
    allow_delegation=False,
    max_iter=10,
    verbose=False,
)

print("Fast agent LLM:", agent_fast.llm.model)
print("Deep agent LLM :", agent_deep.llm.model)
print("Same model?    :", agent_fast.llm.model == agent_deep.llm.model)

## 4. `max_iter` -- preventing infinite loops

Agents iterate: they reason, possibly call a tool, observe the result, and
reason again. `max_iter` caps this loop. If the agent has not produced a final
answer after `max_iter` iterations, CrewAI forces a stop and returns whatever
the agent has so far.

- **Too low** (e.g. 1): agent may not finish simple tasks.
- **Too high** (e.g. 50): agent wastes time on hopeless tasks.
- **Sweet spot**: 5-15 for most tasks; increase only if tools are slow.

In [ ]:
# Demonstrate max_iter with a trivial task.
iter_agent = Agent(
    role="Counter",
    goal="Count to 3, then stop.",
    backstory="You follow instructions precisely.",
    llm=llm,
    allow_delegation=False,
    max_iter=2,          # Very low -- forces early stop.
    verbose=False,
)

from crewai import Task, Crew, Process

iter_task = Task(
    description="Count from 1 to 3, listing each number on its own line.",
    expected_output="Three lines, one number each.",
    agent=iter_agent,
)

iter_crew = Crew(
    agents=[iter_agent],
    tasks=[iter_task],
    process=Process.sequential,
    verbose=False,
)

try:
    result = iter_crew.kickoff()
    print("Result (max_iter=2):", result.raw[:200])
except Exception as e:
    print("[demo skipped]", e)

## 5. `allow_delegation` -- agent-to-agent handoff

When `allow_delegation=True` (the default), an agent can hand off sub-tasks to
other agents in the crew. This is CrewAI's built-in hierarchical pattern. The
agent uses an internal "delegate" tool to ask another agent for help.

- Set to `False` when the agent should work independently.
- Set to `True` for manager/coordinator agents that orchestrate others.

> **Pro tip:** delegation adds latency (extra LLM calls). Only enable it when
> your workflow genuinely needs agent-to-agent communication.

In [ ]:
delegator = Agent(
    role="Project Manager",
    goal="Coordinate the team to complete the task.",
    backstory="You delegate effectively to specialists.",
    llm=llm,
    allow_delegation=True,          # Can hand off to other agents.
    verbose=False,
)

specialist = Agent(
    role="Data Analyst",
    goal="Analyze data and report findings.",
    backstory="You are a careful data analyst.",
    llm=llm,
    allow_delegation=False,         # Works independently.
    verbose=False,
)

deleg_task = Task(
    description="Analyze the trend: AI funding grew 40 percent in 2025.",
    expected_output="A 2-3 sentence analysis of the trend.",
    agent=delegator,                # Manager will delegate to specialist.
)

deleg_crew = Crew(
    agents=[delegator, specialist],
    tasks=[deleg_task],
    process=Process.sequential,
    verbose=False,
)

try:
    deleg_result = deleg_crew.kickoff()
    print("Delegation result:", deleg_result.raw[:300])
except Exception as e:
    print("[demo skipped]", e)

## 6. `verbose` vs `reasoning` -- debugging output

| Flag       | What it prints                                    |
|------------|---------------------------------------------------|
| `verbose`  | Agent/task start/end, tool calls, final output.   |
| `reasoning`| Full chain-of-thought thought process at each step.|

- Use `verbose=True` during development to see the execution flow.
- Use `reasoning=True` when you need to debug *why* an agent made a decision.
- In production, set both to `False` for clean output.

In [ ]:
# verbose=True agent -- shows execution steps.
verbose_agent = Agent(
    role="Echo",
    goal="Repeat back what you were told.",
    backstory="You repeat things.",
    llm=llm,
    allow_delegation=False,
    verbose=True,       # Full step-by-step logging.
    reasoning=False,
)

echo_task = Task(
    description="Say: 'The quick brown fox jumps over the lazy dog.'",
    expected_output="The exact sentence quoted above.",
    agent=verbose_agent,
)

echo_crew = Crew(
    agents=[verbose_agent],
    tasks=[echo_task],
    process=Process.sequential,
    verbose=True,       # Crew-level verbosity too.
)

try:
    echo_result = echo_crew.kickoff()
    print("\nFinal output:", echo_result.raw)
except Exception as e:
    print("[demo skipped]", e)

## 7. Config approach 1 -- Python constructor (recommended)

The Python constructor is the most common approach. It gives you full IDE
autocompletion, type checking, and inline documentation. This is what most
CrewAI projects use.

In [ ]:
python_agent = Agent(
    role="Code Reviewer",
    goal="Review Python code for bugs and style issues.",
    backstory=(
        "You are a senior Python developer with 15 years of experience. "
        "You catch subtle bugs and enforce PEP 8."
    ),
    llm=llm,
    tools=[],
    allow_delegation=False,
    max_iter=5,
    verbose=False,
    reasoning=False,
)

print("Python-config agent:", python_agent.role)

## 8. Config approach 2 -- JSON dictionary

You can define an agent as a plain dictionary and unpack it into `Agent()`.
This is useful for dynamic agent creation from config files or databases.

In [ ]:
agent_config = {
    "role": "Translator",
    "goal": "Translate English text to French accurately.",
    "backstory": (
        "You are a native French speaker with a background in "
        "technical translation."
    ),
    "llm": llm,               # LLM must be an object, not a string in this approach.
    "tools": [],
    "allow_delegation": False,
    "max_iter": 3,
    "verbose": False,
    "reasoning": False,
}

json_agent = Agent(**agent_config)     # Unpack the dict into keyword arguments.
print("JSON-config agent:", json_agent.role)
print("  Goal:", json_agent.goal[:50])

## 9. Config approach 3 -- loading from a JSON file

For larger projects, store agent definitions in a JSON file and load them at
runtime. CrewAI also supports YAML config files natively via `Agent.from_yaml()`,
but the JSON dict approach works everywhere without extra dependencies.

Below we write a temp JSON file, load it, and create an agent from it.

In [ ]:
import tempfile                        # For a safe temp file path.

config_data = {
    "role": "Summarizer",
    "goal": "Condense long texts into 3 bullet points.",
    "backstory": "You are an expert at extracting key information.",
    "allow_delegation": False,
    "max_iter": 3,
    "verbose": False,
}

# Write the config to a temp JSON file.
config_path = Path(tempfile.gettempdir()) / "agent_config.json"
config_path.write_text(json.dumps(config_data, indent=2))
print("Wrote config to:", config_path)

# Load and create agent.
loaded_config = json.loads(config_path.read_text())

# For LLM, we inject the object since JSON cannot store Python objects.
loaded_config["llm"] = llm
loaded_config["tools"] = []

file_agent = Agent(**loaded_config)
print("File-config agent:", file_agent.role)
print("  Goal:", file_agent.goal[:50])

# Cleanup.
config_path.unlink(missing_ok=True)

## 10. Side-by-side comparison of the three approaches

| Approach        | Pros                                | Cons                              |
|-----------------|-------------------------------------|-----------------------------------|
| Python constructor | IDE autocompletion, type safety | Not externalizable to files       |
| JSON dict       | Dynamic, database-friendly          | No type checking, LLM is special |
| JSON file       | Version-control friendly, modular   | File I/O overhead, same LLM note  |

All three produce the same `Agent` object. Choose based on your project:
- **Prototyping**: Python constructor.
- **Config-driven systems**: JSON/YAML file.
- **Dynamic agent creation**: JSON dict at runtime.

In [ ]:
# Verify all three produce functionally identical agents.
print("All three agents created successfully:")
print("  1. python_agent.role  =", repr(python_agent.role))
print("  2. json_agent.role    =", repr(json_agent.role))
print("  3. file_agent.role    =", repr(file_agent.role))

## 11. Key takeaways

- `Agent()` takes role, goal, backstory, llm, tools, allow_delegation, max_iter,
  verbose, and reasoning.
- Each agent can use a **different LLM** -- assign `ChatOllama(model="...")` or
  any LangChain-compatible model.
- `max_iter` prevents infinite loops; default is 15, tune per task complexity.
- `allow_delegation=True` enables agent-to-agent handoff (adds latency).
- `verbose=True` logs execution flow; `reasoning=True` logs chain-of-thought.
- Three config approaches: Python constructor (recommended), JSON dict, JSON file.
- Next module: Tasks, processes, async execution, and callbacks.